In [20]:
import sys
print(sys.executable)

d:\aditya\RAG_Chatbot_Amlgolabs\rag_en\Scripts\python.exe


In [21]:
from pathlib import Path
import re
import json
import fitz  # PyMuPDF
import pandas as pd

In [22]:
PDF_PATH = Path(r"C:\Users\adity\Downloads\AI Training Document.pdf")   
OUTPUT_DIR = Path("chunks")
OUTPUT_DIR.mkdir(exist_ok=True)

print("PDF exists:", PDF_PATH.exists())
print("Output dir:", OUTPUT_DIR.resolve())

PDF exists: True
Output dir: D:\aditya\RAG_Chatbot_Amlgolabs\notebooks\chunks


In [23]:
def extract_pdf_pages(pdf_path: Path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        pages.append({
            "page": page_num,
            "text": text
        })

    doc.close()
    return pages

pages = extract_pdf_pages(PDF_PATH)
print("Total pages:", len(pages))
print("Sample page text:\n")
print(pages[0]["text"][:1500])

Total pages: 20
Sample page text:

User Agreement 
1. Introduction 
This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms 
posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms 
on which eBay offers you access to and use of our Services. You can find an overview of our policies 
here. The Mobile Application Terms of Use, all policies, and additional terms posted on and in our 
Services are incorporated into this User Agreement. You agree to comply with all terms of this User 
Agreement when accessing or using our Services. 
The entity you are contracting with is: eBay Inc., 2025 Hamilton Ave., San Jose, CA 95125, if you 
reside in the United States; eBay (UK) Limited, 1 More London Place, London, SE1 2AF, United 
Kingdom, if you reside in the United Kingdom; eBay GmbH, Albert-Einstein-Ring 2-6, 14532 
Kleinmachnow, Germany, if you reside in the European Union; eBay Canada Limited, 240 Richmond

In [24]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"-\s*\n\s*", "", text)          # join hyphenated line breaks
    text = re.sub(r"\n+", "\n", text)              # collapse repeated newlines
    text = re.sub(r"[ \t]+", " ", text)            # collapse spaces/tabs
    text = re.sub(r"\n\s+", "\n", text)            # trim space after newlines
    text = text.strip()
    return text

In [25]:
def split_into_sentences(text: str):
    text = clean_text(text)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

In [26]:
def chunk_sentences(sentences, target_words=200, overlap_words=50):
    chunks = []
    current_words = []

    for sentence in sentences:
        sent_words = sentence.split()
        if not sent_words:
            continue

        # If sentence itself is too long, split it safely
        while len(sent_words) > target_words:
            part = sent_words[:target_words]
            if part:
                chunks.append(" ".join(part).strip())
            sent_words = sent_words[target_words - overlap_words:] if overlap_words else sent_words[target_words:]

        # Normal case: add sentence to current chunk
        if len(current_words) + len(sent_words) <= target_words:
            current_words.extend(sent_words)
        else:
            if current_words:
                chunks.append(" ".join(current_words).strip())
                current_words = current_words[-overlap_words:] if overlap_words else []
            current_words.extend(sent_words)

    if current_words:
        chunks.append(" ".join(current_words).strip())

    return chunks

In [27]:
all_chunks = []

for page_data in pages:
    raw_text = page_data["text"]
    cleaned = clean_text(raw_text)
    sentences = split_into_sentences(cleaned)
    page_chunks = chunk_sentences(sentences, target_words=200, overlap_words=50)

    for idx, chunk in enumerate(page_chunks, start=1):
        all_chunks.append({
            "chunk_id": len(all_chunks) + 1,
            "page": page_data["page"],
            "page_chunk": idx,
            "text": chunk,
            "word_count": len(chunk.split())
        })

print("Total chunks:", len(all_chunks))
print("First chunk sample:\n")
print(all_chunks[0]["text"][:1200])

Total chunks: 83
First chunk sample:

User Agreement 1. Introduction This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms on which eBay offers you access to and use of our Services. You can find an overview of our policies here. The Mobile Application Terms of Use, all policies, and additional terms posted on and in our Services are incorporated into this User Agreement. You agree to comply with all terms of this User Agreement when accessing or using our Services.


In [28]:
def merge_small_chunks(chunks, min_words=100):
    merged = []
    
    for chunk in chunks:
        if merged and len(chunk.split()) < min_words:
            merged[-1] += " " + chunk
        else:
            merged.append(chunk)
    
    return merged

In [ ]:
chunks = merge_small_chunks([item['text'] for item in all_chunks])

NameError: name 'chunks' is not defined

In [14]:
word_counts = [c["word_count"] for c in all_chunks]

print("Min words:", min(word_counts))
print("Max words:", max(word_counts))
print("Avg words:", sum(word_counts) / len(word_counts))

Min words: 92
Max words: 216
Avg words: 169.87951807228916


In [15]:
for item in all_chunks[:5]:
    print(f"\n--- Chunk {item['chunk_id']} | Page {item['page']} | Words {item['word_count']} ---")
    print(item["text"][:1500])


--- Chunk 1 | Page 1 | Words 96 ---
User Agreement 1. Introduction This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms on which eBay offers you access to and use of our Services. You can find an overview of our policies here. The Mobile Application Terms of Use, all policies, and additional terms posted on and in our Services are incorporated into this User Agreement. You agree to comply with all terms of this User Agreement when accessing or using our Services.

--- Chunk 2 | Page 1 | Words 196 ---
Services. You can find an overview of our policies here. The Mobile Application Terms of Use, all policies, and additional terms posted on and in our Services are incorporated into this User Agreement. You agree to comply with all terms of this User Agreement when accessing or using our Services. The entity you are contracting with is: eBay I

In [16]:
# Save as JSONL
jsonl_path = OUTPUT_DIR / "chunks.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for item in all_chunks:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Save as CSV
csv_path = OUTPUT_DIR / "chunks.csv"
pd.DataFrame(all_chunks).to_csv(csv_path, index=False, encoding="utf-8")

print("Saved:", jsonl_path)
print("Saved:", csv_path)

Saved: chunks\chunks.jsonl
Saved: chunks\chunks.csv


In [17]:
df = pd.DataFrame(all_chunks)
print(df.head())
print("\nTotal rows:", len(df))

   chunk_id  page  page_chunk  \
0         1     1           1   
1         2     1           2   
2         3     1           3   
3         4     1           4   
4         5     1           5   

                                                text  word_count  
0  User Agreement 1. Introduction This User Agree...          96  
1  Services. You can find an overview of our poli...         196  
2  Marketplaces GmbH, Helvetiastrasse 15/17, CH-3...         136  
3  contains an Agreement to Arbitrate which will,...         183  
4  jury trial. 2. About eBay eBay is a marketplac...         173  

Total rows: 83
